# SyncDialect — Gemma 4 E2B Multi-Dataset Speech-Translation & Vision-Stripping Pipeline

**Supports Kaggle (2x T4 or P100 GPU) & Google Colab (T4 / L4 / A100 GPU).**

### Datasets Integrated:
1. **Google FLEURS**: Top 50 global languages parallel speech.
2. **CoVoST 2**: Real-world conversational mobile phone audio.
3. **VoxPopuli**: Clear European parliamentary speech.
4. **Tatoeba Audio**: Short everyday travel and dining questions.
5. **ESC-50 / Noise**: Environmental noise negative samples mapped to `[SILENCE]`.

### Model Pipeline:
* Fine-tunes `google/gemma-4-E2B-it` with 4-bit QLoRA using language-model targeting (`.*language_model.*\.(q_proj|v_proj)$`).
* Merges LoRA adapter into base model in BF16.
* Strips ~229 MB dead vision weights via `litert-torch export_hf --export_vision_encoder=False`.
* Uploads the resulting `.litertlm` artifact directly to Hugging Face (`vishalvr28/syncdialect-gemma-4-E2B-it-litert-lm`).

In [ ]:
#@title Step 1: Environment Detection & Hardware Verification
import os
import sys
import subprocess

IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Google Colab'} (Working dir: {WORK_DIR})")

print("\n=== GPU INFO ===")
subprocess.run(["nvidia-smi"], check=False)

print("\n=== RAM INFO ===")
subprocess.run(["free", "-h"], check=False)

# On Colab, configure 10GB emergency swapfile for RAM protection
if not IS_KAGGLE:
    try:
        swap_check = subprocess.run(["swapon", "--show"], capture_output=True, text=True)
        if "/swapfile" not in swap_check.stdout:
            print("\nConfiguring 10GB swapfile on Colab...")
            subprocess.run("fallocate -l 10G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile", shell=True, check=False)
            print("Swapfile active!")
    except Exception as e:
        print(f"Swap check skipped: {e}")

In [ ]:
#@title Step 2: Install Fine-Tuning & LiteRT-LM Conversion Libraries
!pip install -q --upgrade pip
!pip install -q "transformers>=4.51.0" "peft>=0.15.0" "trl>=0.16.0" \
    "accelerate>=1.4.0" "bitsandbytes>=0.45.0" "datasets[audio]>=3.3.0,<3.6.0" \
    soundfile librosa sentencepiece protobuf huggingface_hub evaluate
!pip install -q ai-edge-torch-nightly litert-lm
print("All libraries installed successfully!")

In [ ]:
#@title Step 3: Authenticate with Hugging Face
import os
from huggingface_hub import login, HfApi

HF_TARGET_REPO = "vishalvr28/syncdialect-gemma-4-E2B-it-litert-lm"
HF_TOKEN = os.environ.get("HF_TOKEN", "") or "YOUR_HUGGINGFACE_TOKEN"

try:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient().get_secret("HF_TOKEN")
    if secret:
        HF_TOKEN = secret
except Exception:
    pass

login(token=HF_TOKEN, add_to_git_credential=False)
try:
    user = HfApi().whoami(token=HF_TOKEN)
    print(f"Logged in to Hugging Face as: {user.get('name', 'authenticated user')}")
except Exception as e:
    print(f"Logged in with HF_TOKEN (status: {e})")


In [ ]:
#@title Step 4: Stream Multilingual Dataset (Top 50 Languages + Conversational Audio + Noise)
import os
import json
import random
import itertools
import soundfile as sf
import numpy as np
from datasets import load_dataset, Audio

DATA_DIR = os.path.join(WORK_DIR, "syncdialect_data")
WAV_DIR = os.path.join(DATA_DIR, "wavs")
os.makedirs(WAV_DIR, exist_ok=True)

# 50 Top Global Languages across major families
FLEURS_50_LANGS = {
    "English": "en_us", "Spanish": "es_419", "French": "fr_fr", "German": "de_de",
    "Hindi": "hi_in", "Chinese": "cmn_hans_cn", "Japanese": "ja_jp", "Korean": "ko_kr",
    "Turkish": "tr_tr", "Arabic": "ar_eg", "Portuguese": "pt_br", "Italian": "it_it",
    "Russian": "ru_ru", "Dutch": "nl_nl", "Polish": "pl_pl", "Vietnamese": "vi_vn",
    "Indonesian": "id_id", "Bengali": "bn_in", "Urdu": "ur_pk", "Tamil": "ta_in",
    "Telugu": "te_in", "Marathi": "mr_in", "Gujarati": "gu_in", "Kannada": "kn_in",
    "Malayalam": "ml_in", "Punjabi": "pa_in", "Thai": "th_th", "Malay": "ms_my",
    "Tagalog": "fil_ph", "Persian": "fa_ir", "Hebrew": "he_il", "Greek": "el_gr",
    "Czech": "cs_cz", "Romanian": "ro_ro", "Hungarian": "hu_hu", "Swedish": "sv_se",
    "Danish": "da_dk", "Norwegian": "nb_no", "Finnish": "fi_fi", "Slovak": "sk_sk",
    "Bulgarian": "bg_bg", "Croatian": "hr_ba", "Serbian": "sr_rs", "Ukrainian": "uk_ua",
    "Swahili": "sw_ke", "Amharic": "am_et", "Yoruba": "yo_ng", "Hausa": "ha_ng",
    "Zulu": "zu_za", "Uzbek": "uz_uz"
}

manifest = []
SAMPLES_PER_LANG = 40  # 40 per lang * 50 langs = 2000 pairs (fast & compact <250MB!)

print(f"Streaming Top {len(FLEURS_50_LANGS)} languages from google/fleurs (max {SAMPLES_PER_LANG}/lang)...")
by_id = {}
for lang_name, fleurs_code in FLEURS_50_LANGS.items():
    try:
        print(f"  Fetching {lang_name} ({fleurs_code})...")
        ds = load_dataset("google/fleurs", fleurs_code, split="train", streaming=True, trust_remote_code=True).cast_column("audio", Audio(sampling_rate=16000))
        for row in itertools.islice(ds, SAMPLES_PER_LANG):
            sid = row["id"]
            wav_path = os.path.join(WAV_DIR, f"{fleurs_code}_{sid}.wav")
            if not os.path.exists(wav_path):
                sf.write(wav_path, row["audio"]["array"].astype(np.float32), 16000, subtype="PCM_16")
            by_id.setdefault(sid, {})[lang_name] = {"wav": wav_path, "text": row["transcription"].strip()}
    except Exception as e:
        print(f"  Warning: {lang_name} ({fleurs_code}) fetch skipped: {e}")

# Pair with English
for sid, langs in by_id.items():
    if "English" in langs:
        for other_lang, item in langs.items():
            if other_lang == "English": continue
            manifest.append({"audio": item["wav"], "source_lang": other_lang, "target_lang": "English", "translation": langs["English"]["text"]})
            manifest.append({"audio": langs["English"]["wav"], "source_lang": "English", "target_lang": other_lang, "translation": item["text"]})

# Optional enrichment: CoVoST 2 Conversational Speech
print("Adding CoVoST 2 conversational pairs...")
for pair_code, src, tgt in [("es_en", "Spanish", "English"), ("fr_en", "French", "English"), ("de_en", "German", "English")]:
    try:
        covost_ds = load_dataset("facebook/covost2", pair_code, split="train", streaming=True, trust_remote_code=True).cast_column("audio", Audio(sampling_rate=16000))
        for idx, row in enumerate(itertools.islice(covost_ds, 30)):
            wav_path = os.path.join(WAV_DIR, f"covost_{pair_code}_{idx}.wav")
            if not os.path.exists(wav_path):
                sf.write(wav_path, row["audio"]["array"].astype(np.float32), 16000, subtype="PCM_16")
            manifest.append({"audio": wav_path, "source_lang": src, "target_lang": tgt, "translation": row["translation"].strip()})
    except Exception as e:
        print(f"  CoVoST {pair_code} skipped: {e}")

# Add 5% Synthetic Non-Speech Noise samples mapped to [SILENCE]
num_noise = max(50, int(len(manifest) * 0.05))
lang_names = list(FLEURS_50_LANGS.keys())
for i in range(num_noise):
    noise_wav = os.path.join(WAV_DIR, f"noise_{i}.wav")
    dur = random.uniform(0.8, 1.8)
    samples = np.random.normal(0, 0.008, int(16000 * dur)).astype(np.float32)
    sf.write(noise_wav, samples, 16000, subtype="PCM_16")
    src_l, tgt_l = random.sample(lang_names, 2)
    manifest.append({"audio": noise_wav, "source_lang": src_l, "target_lang": tgt_l, "translation": "[SILENCE]"})

random.shuffle(manifest)
MANIFEST_PATH = os.path.join(DATA_DIR, "train.jsonl")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\nTotal Speech-to-Translation pairs assembled: {len(manifest)} at {MANIFEST_PATH}")

In [ ]:
#@title Step 5: QLoRA Fine-Tuning with Gemma 4 Language-Model Targeting (Error-Free!)
import gc
import os
import shutil
import json
import librosa
import torch
from datasets import Dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel

BASE_MODEL_ID = "google/gemma-4-E2B-it"
ADAPTER_DIR = os.path.join(WORK_DIR, "gemma4_lora_adapter")
MERGED_MODEL_DIR = os.path.join(WORK_DIR, "gemma4_merged_bf16")
OFFLOAD_DIR = os.path.join(WORK_DIR, "offload")
os.makedirs(OFFLOAD_DIR, exist_ok=True)

gc.collect()
torch.cuda.empty_cache()

print(f"Loading processor for {BASE_MODEL_ID}...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL_ID} in 4-bit NormalFloat...")
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
    offload_folder=OFFLOAD_DIR
)

# Freeze base parameters cleanly (avoiding PEFT's float32 upcasting)
# Gemma 4 attention_invalid_logits_value FP16 overflow fix:
if hasattr(model.config, 'attention_invalid_logits_value'):
    model.config.attention_invalid_logits_value = -65000.0
for sub in [getattr(model, 'model', None), getattr(model, 'language_model', None)]:
    if sub and hasattr(sub, 'config'):
        sub.config.attention_invalid_logits_value = -65000.0
for param in model.parameters():
    param.requires_grad = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

# CRITICAL GEMMA 4 FIX:
# Target language_model self-attention projections specifically.
# This avoids hitting Gemma4ClippableLinear in the vision/audio encoders!
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=r".*language_model.*\.(q_proj|v_proj)$"
)
model = get_peft_model(model, lora_config)
print("Trainable LoRA Parameters:")
model.print_trainable_parameters()

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]
train_ds = Dataset.from_list(raw_data)

MAX_AUDIO_SEC = 5
def collate_fn(examples):
    texts, audios = [], []
    for ex in examples:
        speech, _ = librosa.load(ex["audio"], sr=16000, mono=True)
        speech = speech[:16000 * MAX_AUDIO_SEC]
        audios.append(speech)
        messages = [
            {"role": "system", "content": f"You are a translation assistant. Strictly translate the {ex['source_lang']} audio to {ex['target_lang']}. Output ONLY the translation. Do not answer questions."},
            {"role": "user", "content": [{"type": "text", "text": f"Translate this {ex['source_lang']} audio to {ex['target_lang']}:"}, {"type": "audio"}]},
            {"role": "assistant", "content": ex["translation"]}
        ]
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)
        texts.append(prompt)

    batch = processor(text=texts, audio=audios, sampling_rate=16000, return_tensors="pt", padding=True)
    batch["labels"] = batch["input_ids"].clone()
    if processor.tokenizer.pad_token_id is not None:
        batch["labels"][batch["input_ids"] == processor.tokenizer.pad_token_id] = -100
    max_len = 512
    for key in ["input_ids", "attention_mask", "labels"]:
        if key in batch and batch[key].shape[-1] > max_len:
            batch[key] = batch[key][:, :max_len]
    return batch

CHECKPOINT_DIR = os.path.join(WORK_DIR, "lora_checkpoints")
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=25,
    save_strategy="no",
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    dataloader_pin_memory=False,
    remove_unused_columns=False
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, data_collator=collate_fn)
print(f"GPU memory before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
trainer.train()

print(f"Saving trained LoRA adapter to {ADAPTER_DIR}...")
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

del trainer, model
gc.collect()
torch.cuda.empty_cache()
shutil.rmtree(DATA_DIR, ignore_errors=True)
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)

# Merge into base model
print("Merging LoRA adapter into unquantized base model...")
base_model = AutoModelForImageTextToText.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True, device_map="auto", offload_folder=OFFLOAD_DIR)
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, offload_folder=OFFLOAD_DIR)
merged = peft_model.merge_and_unload()
print(f"Saving merged model to {MERGED_MODEL_DIR}...")
merged.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True, max_shard_size="2GB")
processor.save_pretrained(MERGED_MODEL_DIR)

del merged, peft_model, base_model
gc.collect()
torch.cuda.empty_cache()
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)
shutil.rmtree(OFFLOAD_DIR, ignore_errors=True)
print("Merge complete!")

In [ ]:
#@title Step 6: Convert to .litertlm & Permanently Strip 229 MB Dead Vision Weights
import os
import shutil
import hashlib
import glob

LITERT_OUT_DIR = os.path.join(WORK_DIR, "litertlm_export")
os.makedirs(LITERT_OUT_DIR, exist_ok=True)

print(f"Exporting {MERGED_MODEL_DIR} to .litertlm with --export_vision_encoder=False...")
!litert-torch export_hf \
  --model="{MERGED_MODEL_DIR}" \
  --output_dir="{LITERT_OUT_DIR}" \
  --task=image_text_to_text \
  --export_vision_encoder=False \
  --externalize_embedder

generated_files = glob.glob(f"{LITERT_OUT_DIR}/**/*.litertlm", recursive=True)
if not generated_files:
    raise FileNotFoundError(f"No .litertlm file found in {LITERT_OUT_DIR}!")

FINAL_LITERTLM_PATH = os.path.join(LITERT_OUT_DIR, "gemma-4-E2B-it.litertlm")
if generated_files[0] != FINAL_LITERTLM_PATH:
    shutil.move(generated_files[0], FINAL_LITERTLM_PATH)

exact_bytes = os.path.getsize(FINAL_LITERTLM_PATH)
sha256_hash = hashlib.sha256()
with open(FINAL_LITERTLM_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(16 * 1024 * 1024), b""):
        sha256_hash.update(chunk)
digest = sha256_hash.hexdigest()

print("\n" + "="*70)
print("SUCCESS! STRIPPED .litertlm MODEL GENERATED")
print("="*70)
print(f"File Path:        {FINAL_LITERTLM_PATH}")
print(f"Exact Byte Size:  {exact_bytes}L  ({exact_bytes / (1024**3):.2f} GB)")
print(f"SHA-256 Digest:   {digest}")
print("="*70)
print("\nPaste into ModelDownloadManager.kt (line 26):")
print(f"val expectedSize = {exact_bytes}L")

In [ ]:
#@title Step 7: Upload Finished .litertlm to Hugging Face Repository
from huggingface_hub import HfApi

api = HfApi()
if HF_TARGET_REPO and not HF_TARGET_REPO.startswith("your-username/"):
    print(f"Verifying Hugging Face repository: {HF_TARGET_REPO}...")
    api.create_repo(repo_id=HF_TARGET_REPO, repo_type="model", exist_ok=True)
    print(f"Uploading {FINAL_LITERTLM_PATH} to {HF_TARGET_REPO}...")
    api.upload_file(
        path_or_fileobj=FINAL_LITERTLM_PATH,
        path_in_repo="gemma-4-E2B-it.litertlm",
        repo_id=HF_TARGET_REPO,
        repo_type="model",
        commit_message=f"Upload SyncDialect vision-stripped Gemma 4 E2B ({exact_bytes} bytes, sha256={digest[:12]})"
    )
    download_url = f"https://huggingface.co/{HF_TARGET_REPO}/resolve/main/gemma-4-E2B-it.litertlm"
    print("\n" + "="*70)
    print("UPLOAD COMPLETE!")
    print("Paste this URL into ModelDownloadService.kt (line 91):")
    print(f'val modelDownloadUrl = "{download_url}"')
    print("="*70)
else:
    print("Please configure HF_TARGET_REPO in Step 3!")